In [7]:
import random

In [ ]:
RANKS = "23456789TJQKA"
SUITS = "♠♥♣♦"


def newDeck():
    deck = [f"{rank}{suit}" for rank in RANKS for suit in SUITS]
    random.shuffle(deck)
    return deck


class Player:
    def __init__(self, name, chips):
        self.name = name
        self.chips = chips
        self.hand = []
        self.currentBet = 0
        self.isFolded = False
        self.isAllIn = False
        self.hasActedThisRound = False

    def resetHand(self):
        self.hand = []
        self.currentBet = 0
        self.isFolded = False
        self.isAllIn = False
        self.hasActedThisRound = False


class PokerGame:
    def __init__(self, players):
        self.players = [Player(name, chips) for name, chips in players]

        self.smallBlind = 1
        self.bigBlind = 2

        self.deck = []
        self.communityCards = []

        self.pot = 0
        self.currentPlayerPosition = 0

        self.gameState = "null"
        self.maxBet = 0

    def resetBettingRound(self):
        self.maxBet = 0
        for player in self.players:
            player.currentBet = 0
            player.hasActedThisRound = False

    def dealHoleCards(self):
        for _ in range(2):
            for player in self.players:
                player.hand.append(self.deck.pop(0))

    def dealFlop(self):
        self.resetBettingRound()
        self.deck.pop(0)  # Burn card
        for _ in range(3):
            self.communityCards.append(self.deck.pop(0))0
            

    def dealTurn(self):
        self.resetBettingRound()
        self.deck.pop(0)  # Burn card
        self.communityCards.append(self.deck.pop(0))

    def dealRiver(self):
        self.resetBettingRound()
        self.deck.pop(0)  # Burn card
        self.communityCards.append(self.deck.pop(0))

    def currentPlayer(self):
        return self.players[self.currentPlayerPosition]

    def advancePlayer(self):
        self.currentPlayerPosition = (self.currentPlayerPosition + 1) % len(self.players)

    def activePlayers(self):
        return [player for player in self.players if not player.isFolded]

    def printSeparator(self, title):
        print("=" * 70)
        if title:
            print(title)
            print("=" * 70)

    def printPlayerStatus(self):
        print("PLAYER STATUS")
        print("Name                 | Chips    | Bet      | Hand")
        print("---------------------+----------+----------+----------------")
        for player in self.players:
            print(f"{player.name:<20} | ${player.chips:<7} | ${player.currentBet:<7} | {player.hand}")
        print("---------------------+----------+----------+----------------")

    def printRoundStatus(self):
        print(f"Pot: ${self.pot} | Current highest bet: ${self.maxBet}")
        print(f"Community cards: {self.communityCards}\n")

    def postBlinds(self):
        sb = self.players[0]
        bb = self.players[1]

        sb.chips -= self.smallBlind
        sb.currentBet += self.smallBlind
        sb.hasActedThisRound = True

        bb.chips -= self.bigBlind
        bb.currentBet += self.bigBlind
        bb.hasActedThisRound = True

        self.pot += self.smallBlind + self.bigBlind
        self.maxBet = self.bigBlind

        self.printSeparator("Posting blinds")

        print(f"Player {sb.name} posted small blind of {self.smallBlind}.")
        print(f"Player {bb.name} posted big blind of {self.bigBlind}.")

        self.advancePlayer()
        self.advancePlayer()
        self.printPlayerStatus()
        self.printRoundStatus()

    def checkValidBet(self, player, bet):
        if bet == -1:
            player.isFolded = True
            return True

        if bet > player.chips:
            print(f"Invalid bet. You only have ${player.chips}. Please enter a valid bet.")
            return False

        if bet < 0:
            print("Invalid bet. Please enter a positive number, or -1 to fold.")
            return False

        if player.currentBet + bet < self.maxBet:
            print(f"Invalid bet. Please enter a bet greater than or equal to ${self.maxBet - player.currentBet}.")
            return False

        if bet == player.chips:
            player.isAllIn = True
            return True

        return True

    def action(self, player):
        print(f"Curent bet is ${self.maxBet}")
        while True:
            try:
                bet = int(input(f"{player.name}, enter your bet, -1 to fold: "))
            except ValueError:
                continue

            if self.checkValidBet(player, bet):
                return bet

    def playerTurn(self):
        player = self.currentPlayer()

        if player.isFolded:
            self.advancePlayer()
            return

        self.printPlayerStatus()
        self.printSeparator(f"{player.name}'s turn")
        self.printRoundStatus()

        print(f"Player hand: {player.hand}")
        bet = self.action(player)

        # If folded this turn
        if bet == -1:
            player.isFolded = True
            player.hasActedThisRound = True
            self.advancePlayer()
            return

        # Apply bet
        player.chips -= bet
        player.currentBet += bet
        self.pot += bet
        player.hasActedThisRound = True

        # If raised
        if player.currentBet > self.maxBet:
            self.maxBet = player.currentBet
            for other in self.players:
                if not other.isFolded:
                    other.hasActedThisRound = False
            player.hasActedThisRound = True

        self.advancePlayer()

    def bettingRoundComplete(self):
        active = self.activePlayers()

        if len(active) <= 1:
            return True

        if not all(player.hasActedThisRound for player in active):
            return False

        return all(((player.currentBet == self.maxBet) or player.isAllIn) for player in active)

    def bettingRound(self):
        while not self.bettingRoundComplete():
            self.playerTurn()

        print("Betting round complete.")

    def startHand(self):
        self.deck = newDeck()
        self.communityCards = []
        self.pot = 0

        for player in self.players:
            player.resetHand()

        self.dealHoleCards()
        self.postBlinds()

        self.gameState = "pre-flop"
        self.bettingRound()

        self.dealFlop()
        self.bettingRound()

        self.dealTurn()
        self.bettingRound()

        self.dealRiver()
        self.bettingRound()


game = PokerGame([("Player1", 1000), ("Player2", 1000), ("Player3", 1000)])
game.startHand()


Posting blinds
Player Player1 posted small blind of 1.
Player Player2 posted big blind of 2.
PLAYER STATUS
Name                 | Chips    | Bet      | Hand
---------------------+----------+----------+----------------
Player1              | $999     | $1       | ['5♦', '4♥']
Player2              | $998     | $2       | ['9♠', '5♥']
Player3              | $1000    | $0       | ['Q♥', 'K♥']
---------------------+----------+----------+----------------
Pot: $3 | Current highest bet: $2
Community cards: []

PLAYER STATUS
Name                 | Chips    | Bet      | Hand
---------------------+----------+----------+----------------
Player1              | $999     | $1       | ['5♦', '4♥']
Player2              | $998     | $2       | ['9♠', '5♥']
Player3              | $1000    | $0       | ['Q♥', 'K♥']
---------------------+----------+----------+----------------
Player3's turn
Pot: $3 | Current highest bet: $2
Community cards: []

Player hand: ['Q♥', 'K♥']
Curent bet is $2
PLAYER STATUS
Name 